# ClinicalTrials.gov (612 k studies × 64)

[ClinicalTrials.gov](https://clinicaltrials.gov/) is a database of privately and
publicly funded clinical studies.  This notebook downloads study metadata via the
[ClinicalTrials.gov API v2](https://clinicaltrials.gov/data-api/api), embeds the brief
title of each study with the
[`minishlab/potion-base-8M`](https://huggingface.co/minishlab/potion-base-8M)
static embedding model (English, 256 dims), and reduces to 64 dims with PCA —
exactly as the Wikipedia notebook does for article titles.

**Terms of use:** ClinicalTrials.gov data are in the public domain; see the
[NLM copyright and disclaimer notice](https://www.nlm.nih.gov/copyright.html) and the
[ClinicalTrials.gov terms and conditions](https://clinicaltrials.gov/about-site/terms-conditions).
Commercial redistribution of the raw data may require a data licence; always consult
the current terms before redistributing.

**Data source:** ClinicalTrials.gov API v2 — paginated JSON, `pageSize=1000`, fields:
NCTId, BriefTitle, Phase, OverallStatus, LeadSponsorClass, Conditions, StartDate,
EnrollmentCount.  Pages are cached in `~/.cache/positron-atlas/datasets/trials/pages/`.
Full download is ~612 k studies across ~613 pages.

**Embedding model:** `minishlab/potion-base-8M` (model2vec, English, MIT licence).

**First run:** downloading all pages takes roughly 15–30 minutes (rate-limited); the
embedding + PCA step adds 5–10 minutes.  Results are cached in
`~/.cache/positron-atlas/datasets/trials/trials.parquet`.
Subsequent runs skip every step that is already cached.

**Quick try:** set `SAMPLE = True` to download only the first 100 pages (~100 k studies,
a few minutes).

**What to try in Atlas:**
- Hover a study to see its title and open it on ClinicalTrials.gov.
- Color by `phase` to see the clinical-trial landscape.
- Color by `sponsor_class` (INDUSTRY vs NIH vs OTHER) to see funding patterns.
- Use the histogram filter on `start_date` to explore when trials started.
- Filter on `enrollment` to focus on large trials.
- Search for *COVID-19*, *diabetes*, or *Alzheimer* to find specific disease areas.
- Annotate clusters corresponding to oncology or cardiovascular trials.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set SAMPLE = True to download only 100 pages (~100 k studies, a few minutes).
# Set SAMPLE = False for the full ~612 k studies (~30 min first run).
SAMPLE = False

PAGE_SIZE    = 1000     # studies per API page
SAMPLE_PAGES = 100      # pages to download in SAMPLE mode
N_COMPONENTS = 64       # PCA output dimensions (f0 … f63)
BATCH_SIZE   = 4096     # titles per embedding batch
PCA_SAMPLE   = 100_000  # titles used to fit PCA
REQUEST_DELAY = 0.1     # seconds between API requests (be polite)

In [ ]:
import json, time, urllib.error, urllib.request
from pathlib import Path

import numpy as np
import polars as pl
from model2vec import StaticModel

CACHE     = Path.home() / ".cache" / "positron-atlas" / "datasets" / "trials"
PAGES_DIR = CACHE / "pages"
TRIALS_PARQUET = CACHE / "trials.parquet"
PCA_CACHE      = CACHE / "pca_components.npz"
PAGES_DIR.mkdir(parents=True, exist_ok=True)

_API_BASE = "https://clinicaltrials.gov/api/v2/studies"
_FIELDS   = (
    "NCTId,BriefTitle,Phase,OverallStatus,LeadSponsorClass,"
    "ConditionBrowseModule,StartDate,EnrollmentCount"
)


def _page_path(n: int) -> Path:
    return PAGES_DIR / f"page_{n:06d}.json"


def _extract_study(study: dict) -> dict:
    """Extract flat fields from a v2 study object."""
    proto  = study.get("protocolSection", {})
    ident  = proto.get("identificationModule", {})
    status = proto.get("statusModule", {})
    design = proto.get("designModule", {})
    sponsor = proto.get("sponsorCollaboratorsModule", {})
    conds  = proto.get("conditionsModule", {})

    phases = design.get("phases", [])
    phase  = phases[0].replace("PHASE", "Phase ") if phases else None

    sponsor_class = (sponsor.get("leadSponsor") or {}).get("class")

    # conditions come from either conditionsModule or the browse mesh
    cond_list = conds.get("conditions", [])
    if not cond_list:
        cond_list = [
            m.get("meshTerm", "")
            for m in (study.get("derivedSection", {})
                      .get("conditionBrowseModule", {})
                      .get("meshes", []))
        ]
    condition = cond_list[0] if cond_list else None

    start_struct = status.get("startDateStruct", {})
    start_date   = start_struct.get("date")           # "YYYY-MM" or "YYYY-MM-DD"
    if start_date and len(start_date) == 7:
        start_date += "-01"                            # normalise to full date

    enroll_info  = design.get("enrollmentInfo") or {}
    enrollment   = enroll_info.get("count")

    return {
        "nct_id":        ident.get("nctId", ""),
        "title":         ident.get("briefTitle", ""),
        "phase":         phase,
        "status":        status.get("overallStatus"),
        "sponsor_class": sponsor_class,
        "conditions":    condition,
        "start_date":    start_date,
        "enrollment":    enrollment,
    }


def _embed_batched(model: StaticModel, titles: list[str]) -> np.ndarray:
    """Embed titles in BATCH_SIZE chunks; return float32 (n, dim) array."""
    out = []
    for i in range(0, len(titles), BATCH_SIZE):
        out.append(model.encode(titles[i : i + BATCH_SIZE]))
    return np.concatenate(out, axis=0).astype(np.float32)

In [ ]:
# ── Phase 1: paginate the API, caching each page as JSON ──────────────────────
# Resume-safe: pages already on disk are skipped; the next page token is read
# from the last cached page.

if TRIALS_PARQUET.exists():
    print("trials.parquet already cached — skipping Phase 1")
else:
    _existing = sorted(PAGES_DIR.glob("page_*.json"))
    _n_start  = len(_existing)
    if _n_start:
        _last_data  = json.loads(_existing[-1].read_text())
        _page_token = _last_data.get("nextPageToken")
        print(f"Resuming from page {_n_start} (token: {_page_token!r})")
    else:
        _page_token = None

    _page_num     = _n_start
    _total_so_far = sum(len(json.loads(p.read_text()).get("studies", [])) for p in _existing)
    _t0           = time.perf_counter()

    while True:
        if SAMPLE and _page_num >= SAMPLE_PAGES:
            break
        if _page_token is None and _page_num > 0:
            break

        _url = f"{_API_BASE}?pageSize={PAGE_SIZE}&fields={_FIELDS}"
        if _page_token:
            _url += f"&pageToken={_page_token}"

        for _attempt in range(5):
            try:
                with urllib.request.urlopen(_url, timeout=30) as _resp:
                    _data = json.loads(_resp.read())
                break
            except (urllib.error.URLError, OSError) as _e:
                _wait = 2 ** _attempt
                print(f"  Retry {_attempt + 1}/5 after {_wait}s ({_e})")
                time.sleep(_wait)
        else:
            raise RuntimeError("API unavailable after 5 retries")

        _page_path(_page_num).write_text(json.dumps(_data))
        _n = len(_data.get("studies", []))
        _total_so_far += _n
        _page_num += 1

        if _page_num % 50 == 0 or _n < PAGE_SIZE:
            _elapsed = time.perf_counter() - _t0
            print(f"  Page {_page_num}: {_total_so_far:,} studies total ({_elapsed:.0f} s)")

        _page_token = _data.get("nextPageToken")
        if _page_token is None:
            break
        time.sleep(REQUEST_DELAY)

    print(f"Phase 1 done: {_page_num} pages, {_total_so_far:,} studies")

In [ ]:
# ── Phase 2: load model once; fit PCA on a sample of titles ───────────────────
if TRIALS_PARQUET.exists():
    print("trials.parquet already cached — skipping Phase 2")
else:
    _all_pages = sorted(PAGES_DIR.glob("page_*.json"))
    if not _all_pages:
        raise RuntimeError("No pages cached; run Phase 1 first.")

    print("Phase 2: loading model...")
    _t0 = time.perf_counter()
    model = StaticModel.from_pretrained("minishlab/potion-base-8M")
    print(f"  Model loaded in {time.perf_counter() - _t0:.1f} s (dim={model.dim})")

    if PCA_CACHE.exists():
        _npz = np.load(PCA_CACHE)
        _pca_mean, _pca_components = _npz["mean"], _npz["components"]
        print(f"Loaded PCA cache: {_pca_components.shape}")
    else:
        # Collect up to PCA_SAMPLE titles for PCA fitting.
        _sample_titles: list[str] = []
        for _p in _all_pages:
            _d = json.loads(_p.read_text())
            for _s in _d.get("studies", []):
                _t = (_s.get("protocolSection", {})
                        .get("identificationModule", {})
                        .get("briefTitle", ""))
                if _t:
                    _sample_titles.append(_t)
            if len(_sample_titles) >= PCA_SAMPLE:
                break

        _rng = np.random.default_rng(42)
        _idx = _rng.choice(len(_sample_titles), min(PCA_SAMPLE, len(_sample_titles)), replace=False)
        _sample = [_sample_titles[int(i)] for i in sorted(_idx)]
        del _sample_titles

        print(f"  Embedding {len(_sample):,} titles for PCA fitting...")
        _t1 = time.perf_counter()
        _sample_embs = _embed_batched(model, _sample)
        _rate = len(_sample) / (time.perf_counter() - _t1)
        print(f"  Embedded in {time.perf_counter() - _t1:.1f} s ({_rate:,.0f} titles/s)")

        _pca_mean = _sample_embs.mean(axis=0)
        _C = (_sample_embs - _pca_mean).astype(np.float32)
        _cov = (_C.T @ _C) / (len(_C) - 1)
        _evals, _evecs = np.linalg.eigh(_cov.astype(np.float64))
        _order = np.argsort(_evals)[::-1]
        _pca_components = _evecs[:, _order[:N_COMPONENTS]].T.astype(np.float32)
        del _C, _cov, _evals, _evecs, _sample_embs

        np.savez(PCA_CACHE, mean=_pca_mean, components=_pca_components)
        print(f"  PCA fitted and cached ({_pca_components.shape})")

In [ ]:
# ── Phase 3: embed all titles, project, assemble DataFrame ────────────────────
if TRIALS_PARQUET.exists():
    print("trials.parquet already cached — loading")
else:
    print("Phase 3: embedding all study titles and assembling DataFrame...")
    _t0 = time.perf_counter()
    _all_pages = sorted(PAGES_DIR.glob("page_*.json"))

    _rows: list[dict] = []
    _titles: list[str] = []

    for _p in _all_pages:
        _d = json.loads(_p.read_text())
        for _s in _d.get("studies", []):
            _r = _extract_study(_s)
            _rows.append(_r)
            _titles.append(_r["title"])

    print(f"  Loaded {len(_rows):,} studies; embedding titles...")
    _t1 = time.perf_counter()
    _embs = _embed_batched(model, _titles)
    _proj = ((_embs - _pca_mean) @ _pca_components.T).astype(np.float32)
    _rate = len(_titles) / (time.perf_counter() - _t1)
    print(f"  Embedded in {time.perf_counter() - _t1:.1f} s ({_rate:,.0f} titles/s)")

    _df = pl.DataFrame({
        "nct_id":        [r["nct_id"]        for r in _rows],
        "title":         [r["title"]         for r in _rows],
        "phase":         [r["phase"]         for r in _rows],
        "status":        [r["status"]        for r in _rows],
        "sponsor_class": [r["sponsor_class"] for r in _rows],
        "conditions":    [r["conditions"]    for r in _rows],
        "start_date":    pl.Series([r["start_date"] for r in _rows]).str.to_date(format="%Y-%m-%d", strict=False),
        "enrollment":    pl.Series([r["enrollment"]  for r in _rows]).cast(pl.Int32, strict=False),
    }).with_columns(
        pl.format("https://clinicaltrials.gov/study/{}", pl.col("nct_id")).alias("url"),
    )

    _feat_df = pl.from_numpy(_proj, schema=[f"f{i}" for i in range(N_COMPONENTS)])
    trials = _df.hstack(_feat_df)

    trials.write_parquet(TRIALS_PARQUET)
    print(f"Saved trials.parquet ({TRIALS_PARQUET.stat().st_size / 1e6:.0f} MB, {len(trials):,} rows) in {time.perf_counter() - _t0:.0f} s")

trials = pl.read_parquet(TRIALS_PARQUET)
trials.shape

## Open it in Atlas

1. In the **Variables** pane, double-click **`trials`** to open it in the Data Explorer.
2. Click **Open in Atlas**, the first button in the Data Explorer's toolbar (top left).

Or run **Atlas: Open DataFrame…** from the Command Palette and pick `trials`.

## Things to try

- **Features only** (`Map → Data`): Atlas includes `enrollment` as a numeric feature
  by default. For a pure title-embedding map, deselect `enrollment` and keep only
  `f0`–`f63`.
- **Color by `phase`.**  Phase 3 and 4 trials cluster together; Phase 1 (first-in-human)
  sits in a broad, dispersed region.
- **Color by `sponsor_class`.**  Industry-sponsored and NIH-funded trials have distinct
  spatial patterns.
- **Histogram filter on `start_date`.**  Slide the range to watch the clinical-trial
  landscape evolve over time.  The COVID-19 surge in 2020–2021 becomes visible.
- **Histogram filter on `enrollment`.**  Set a minimum of 1,000 to focus on large trials.
- **Search** for *COVID-19*, *diabetes*, or *breast cancer* to find disease clusters.
  Try *vaccine*, *placebo*, or *randomized* for trial-design keywords.
- **Turn on cluster labels** to discover disease areas.
- **Annotate** the COVID-19 vaccine cluster (2020 trials) or the large oncology region.